# 05 有限测量、候选选择与器件噪声

本节属于课程进阶模块。代码、教学构造、历史实测、待执行实验分别标注。云执行默认关闭；真实金融数据通过仓库外的私有快照接入。原始行情、逐样本标签、角度、核矩阵、参数与预测不得存入公开Notebook输出。

**来源**：[L001 ICLR正式版](https://proceedings.iclr.cc/paper_files/paper/2024/file/4d9944ab3330fe6af8efb9260aa9f307-Paper-Conference.pdf)、[本课程协议](../../docs/天衍云端复现协议.md)。署名：Julius Zhou、AUFE、Cong Lei。代码MIT，原创教学文字与图表CC BY 4.0。


In [1]:
from pathlib import Path
import sys, os, json
import numpy as np
import pandas as pd
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "课程大纲.md").exists())
MODULE = COURSE / "experiments/qukernet_finance"
sys.path.insert(0, str(MODULE))
PUBLIC = COURSE / "reports/qukernet_cloud_v2"
ENABLE_CLOUD = False
print("默认云执行关闭；当前Notebook不读取凭据、不提交任务。")


默认云执行关闭；当前Notebook不读取凭据、不提交任务。


## 三个不同问题
候选损失差是否能辨认、重复测量是否稳定、失选是否造成实际代价需要分别报告。固定候选池精确概率后的经典重采样可以研究有限shots，但不是云端真实执行，也没有验证测量误差贯穿评分器训练的全过程。

零串未命中时，单项单侧95%上界为$1-0.05^{1/N}$。这不是精确概率为0的证明；多个核元素和多次查看还需处理多重比较。

In [2]:
shots=np.array([32,128,512,4096,100000])
display(pd.DataFrame({"shots":shots,"zero_hits_one_sided_95_upper":1-.05**(1/shots)}))
print("这是解析二项界，没有生成金融或云结果。")

,shots,zero_hits_one_sided_95_upper
0,32,0.089368
1,128,0.023132
2,512,0.005834
3,4096,0.000731
4,100000,0.000030


这是解析二项界，没有生成金融或云结果。


In [3]:
finite=pd.read_csv(PUBLIC/"historical_finite_aggregates.csv")
assert (finite.real_cloud_shots==0).all()
display(finite[["shots","successful","modal_fraction","mean_regret","nominal_shots","real_cloud_shots"]])

,shots,successful,modal_fraction,mean_regret,nominal_shots,real_cloud_shots
0,32,20,0.65,0.001084,14643200,0
1,128,20,0.80,0.000483,58572800,0
2,512,20,1.00,0.000000,234291200,0


## 噪声的位置决定结论
有限测量是抽样随机性，器件噪声改变目标分布。噪声是否影响标签生成、筛选、微调和最终评分必须逐项说明。`noise`参数被客户端接受不证明服务器正确施加了模型；必须用解析可检验的线路检查。

旧候选池精确损失范围小于0.01，故阈值0.01/0.02/0.05下零失选率存在结构性原因，不能单独宣称可靠。新诊断同时画真实间隔、频率、遗憾和失败比例。

**作业**：构造“选择频繁换位但遗憾极小”和“稳定选错且遗憾较大”两个例子。说明为什么同一金融样本不意味着不同线路的测量读数能够共享。